## Demo of vessel density calculation

In [1]:
# Import necessary libraries
from liom_toolkit.segmentation.stats import generate_itk_id_list_of_region, create_filter_image, filter_image_to_region, compute_mask_area
from liom_toolkit.utils import load_node_by_name, load_zarr, dask_client_manager

In [2]:
# Setup dask client.

# Local
dask_client_manager.set_client()

# Remote
#dask_client_manager.set_client("tcp://...")

client = dask_client_manager.get_client()

/home/frans/code/liom-toolkit/.venv/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33703 instead
  warnings.warn(


In [3]:
# Load the zarr datasets. This notebook uses the S23 zarr which has
# atlas, vessels, and mask labels (full pipeline output) on the LSFM server.
image_path = "/data/LSFM/S23/S23.zarr"
nodes = load_zarr(image_path)
# Set the resolution level, 0 is the highest resolution
resolution_level = 2  # level 2 (513x512x512) — level 0 is 23GB

# Load the atlas, image and mask
atlas_node = load_node_by_name(nodes, "atlas")
atlas = client.scatter(atlas_node.data[resolution_level])

vascular_node = load_node_by_name(nodes, "vessels")
vascular = client.scatter(vascular_node.data[resolution_level])

mask_node = load_node_by_name(nodes, "mask")
mask = client.scatter(mask_node.data[resolution_level])


version mismatch: detected: FormatV04, requested: FormatV05


In [4]:
# Define the region of interest
region = "Thalamus"

In [5]:
# Get the region ids which are in the atlas
itk_ids = generate_itk_id_list_of_region(region)

In [6]:
# Create a filter image
filter_image = create_filter_image(atlas, itk_ids)

# Apply the filter image to the vascular image and the mask
filtered_vascular = filter_image_to_region(vascular, filter_image)
filtered_mask = filter_image_to_region(mask, filter_image)

In [7]:
# Compute the areas of both the vessel and the mask
vascular_area = compute_mask_area(filtered_vascular)
mask_area = compute_mask_area(filtered_mask)

In [8]:
# Calculate the vessel density
vessel_density = vascular_area / mask_area
vessel_density

/tmp/ipykernel_237328/3649111622.py:2: RuntimeWarning: invalid value encountered in scalar divide
  vessel_density = vascular_area / mask_area


np.float64(nan)